In [ ]:
"""
Py-Microgrid Hybrid System Simulation Example
-----------------------------------
This example demonstrates how to:
1. Set up a hybrid system simulation
2. Download solar and wind resource data
3. Configure system parameters
4. Run optimization
5. Analyze and save results

Required files:
- Base YAML configuration file
- CSV file containing location data
"""

import os
import pandas as pd
from typing import Dict, List, Any

# Set NREL API key FIRST before any other imports to avoid timing issues
from py_microgrid.utilities.keys import set_developer_nrel_gov_key
set_developer_nrel_gov_key('ZaurwKOnwDUp8rMyNBIxI4XiBo3b7L5oruTi0VX3')

# Required imports (after API key is set)
from py_microgrid.utilities import ConfigManager
from py_microgrid.tools.optimization.system_optimizer import SystemOptimizer  # CORRECTED: Use proper system optimizer
from py_microgrid.tools.optimization import LoadAnalyzer
from py_microgrid.tools.analysis.bos import EconomicCalculator
from py_microgrid.simulation.resource_files import ResourceDataManager

# Initialize resource manager for downloading data
resource_manager = ResourceDataManager(
    api_key='ZaurwKOnwDUp8rMyNBIxI4XiBo3b7L5oruTi0VX3',
    email='hanrong.h99@gmail.com'
)

# Location information
latitude = -33.5265
longitude = 149.1588

# Download resource data
solar_path = resource_manager.download_solar_data(
    latitude=latitude,
    longitude=longitude,
    year="2020" 
)
wind_path = resource_manager.download_wind_data(
    latitude=latitude,
    longitude=longitude,
    start_date="20200101",  
    end_date="20201231"
)

# Load and update YAML configuration
yaml_file_path = "./quick_start_config.yaml"  # Updated path
config_manager = ConfigManager()
config = config_manager.load_yaml_safely(yaml_file_path)

# Update configuration with location and resource files
config['site']['data']['lat'] = latitude
config['site']['data']['lon'] = longitude
config['site']['solar_resource_file'] = solar_path.replace('\\', '/')  # Normalize path separators
config['site']['wind_resource_file'] = wind_path.replace('\\', '/')

# Save updated configuration
config_manager.save_yaml_safely(config, yaml_file_path)

# Initialize components for optimization with hardcoded economic parameters
economic_calculator = EconomicCalculator(
    discount_rate=0.0588,    # 5.88% discount rate (hardcoded)
    project_lifetime=25      # 25 year project lifetime (hardcoded)
)

optimizer = SystemOptimizer(
    yaml_file_path=yaml_file_path,
    economic_calculator=economic_calculator,
    enable_flexible_load=True,  # Set to False to disable flexible load
    max_load_reduction_percentage=0.2  # 20% maximum load reduction (hardcoded)
)

# Define optimization bounds - NOW PROPERLY HANDLES GRID COMPONENT
# Check if grid is enabled to determine bounds
config = config_manager.load_yaml_safely(yaml_file_path)
grid_enabled = config.get('technologies', {}).get('grid', {}).get('enabled', False)

if grid_enabled:
    # 6-component optimization: PV, Wind, Battery kWh, Battery kW, Genset, Grid
    bounds = [
        (5000, 50000),    # PV capacity (kW)
        (1, 50),          # Wind turbines (1MW each)
        (5000, 30000),    # Battery capacity (kWh)
        (1000, 10000),    # Battery power (kW)
        (17000, 30000),   # Genset capacity (kW) - backup generator
        (5000, 25000)     # Grid capacity (kW) - grid connection
    ]
    # Initial conditions for 6 components
    initial_conditions = [
        [bound[0] + (bound[1] - bound[0]) * 0.1 for bound in bounds]
    ]
else:
    # 5-component optimization: PV, Wind, Battery kWh, Battery kW, Genset
    bounds = [
        (5000, 50000),    # PV capacity (kW)
        (1, 50),          # Wind turbines (1MW each)
        (5000, 30000),    # Battery capacity (kWh)
        (1000, 10000),    # Battery power (kW)
        (17000, 30000)    # Genset capacity (kW) - backup generator
    ]
    # Initial conditions for 5 components
    initial_conditions = [
        [bound[0] + (bound[1] - bound[0]) * 0.1 for bound in bounds]
    ]

print(f"Running optimization with {'6' if grid_enabled else '5'} components")
print(f"Grid enabled: {grid_enabled}")

# Run Nelder-Mead optimization with proper cost model integration
result = optimizer.optimize_system(bounds, initial_conditions)

# Print results with original format
if result:
    print("\nOptimization Results:")
    print(f"PV Capacity: {result['PV Capacity (kW)']:.2f} kW")
    print(f"Wind Turbines: {result['Wind Turbine Capacity (kW)'] / 1000:.0f} x 1MW")
    print(f"Battery Capacity: {result['Battery Energy Capacity (kWh)']:.2f} kWh")
    print(f"Battery Power: {result['Battery Power Capacity (kW)']:.2f} kW")
    print(f"Genset Capacity: {result['Genset Capacity (kW)']:.2f} kW")
    if grid_enabled:
        print(f"Grid Capacity: {result['Grid Capacity (kW)']:.2f} kW")
    print(f"\nLCOE: ${result['System LCOE ($/kWh)']:.4f}/kWh")
    print(f"System Net Present Cost: ${result['System NPC ($)']:,.2f}")
    print(f"CO2 Emissions: {result['Total CO2 emissions (tonne)']:.2f} tonnes")
    print(f"Demand Met: {result['Demand Met Percentage']:.2f}%")
else:
    print("Optimization failed to find a solution")

: 

In [ ]:
# Save optimization results to CSV
if result:
    results_df = pd.DataFrame([result])
    csv_filename = f"optimization_results_{latitude}_{longitude}.csv"
    results_df.to_csv(csv_filename, index=False)
    print(f"✓ Optimization results saved to: {csv_filename}")
else:
    print("✗ No results to save - optimization failed")

print("✓ Quick start example completed successfully!")